# Document processing: report-level sentiment

Runs the single extraction script over the 10-K and 10-Q samples in `input_data/`
and writes report-level polarities to `sentiment_extraction_results/`.

All the logic lives in `code/scripts/`; this notebook runs it.

One procedure for both form types: the filing is split into token-budgeted chunks,
each chunk gets the prompt ending appended, one token position is read out, the
top-p words is matched against the verbalizer, and the chunk scores are averaged
into one row per filing.

Each filing is keyed by `(CIK, FILING_DATE, ACC_NUM, FORM_TYPE)`, so a re-run
resumes exactly where it stopped and merges stay safe.

In [1]:
# !pip install torch
# !pip install langchain_text_splitters
# !pip install transformers

In [2]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "code" / "scripts"))

import pandas as pd

from utils import INPUT_DATA_DIR, SENTIMENT_RESULTS_DIR, clean_mem, set_seed
from reports import discover_reports
from prompts import STRATEGIES
from sentiment_extraction import ExtractionConfig, ScoringBackend, run_extraction

pd.set_option("display.max_columns", None)
set_seed(42)

print("repo:", REPO_ROOT)
print("input:", INPUT_DATA_DIR)
print("output:", SENTIMENT_RESULTS_DIR)
print("strategies:", sorted(STRATEGIES))

repo: /Users/natalya-vasileva/Desktop/manager_sentiment
input: /Users/natalya-vasileva/Desktop/manager_sentiment/input_data
output: /Users/natalya-vasileva/Desktop/manager_sentiment/sentiment_extraction_results
strategies: ['financial_risk', 'growth', 'growth_legacy_causal']


## Sample

File names carry the filing key:
`{FILING_DATE}_{FORM_TYPE}_edgar_data_{CIK}_{ACC_NUM}.txt`.

In [3]:
sample = discover_reports(INPUT_DATA_DIR)

print(sample["FORM_TYPE"].value_counts().to_string())
print()
print("filing years:", sorted(sample["FILING_DATE"].str[:4].unique()))
sample.head()

23:09:22 | INFO    | reports | discovered 100 filing(s) under /Users/natalya-vasileva/Desktop/manager_sentiment/input_data


FORM_TYPE
10-Q    50
10-K    50

filing years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']


,CIK,FILING_DATE,ACC_NUM,FORM_TYPE,SECTION,path
0,101295,20211112,0001171843-21-007870,10-Q,None,/Users/natalya-vasileva/Desktop/manager_sentim...
1,1022408,20220526,0001140361-22-020620,10-K,None,/Users/natalya-vasileva/Desktop/manager_sentim...
2,1046995,20140313,0001013762-14-000233,10-K,None,/Users/natalya-vasileva/Desktop/manager_sentim...
3,1069533,20171208,0001069533-17-000036,10-K,None,/Users/natalya-vasileva/Desktop/manager_sentim...
4,1070412,20150206,0001070412-15-000009,10-K,None,/Users/natalya-vasileva/Desktop/manager_sentim...


## Configuration

`chunk_size_tokens=None` keeps the earlier convention: chunk size is the model
window minus a reserve for the prompt ending (1000 tokens for a causal LM, 100
for a masked LM). Note what that implies per model:

- **ModernBERT-large** (window 8192) chunks long filings.
- **Llama-3.1-8B** (window 131072) leaves ~130k tokens per chunk, so almost
  every filing is a single chunk and the averaging step is a no-op.

Set `chunk_size_tokens` explicitly (e.g. 4096) if the two models should see the
same amount of text per readout.

The repository does not contain model weights. Download Llama 3.1 8B and
ModernBERT-large with the commented commands in `code/scripts/download_models.sh`,
then set `MODEL_ID` to the local directory or to the Hugging Face id.

In [4]:
# MODEL_ID = "meta-llama/Meta-Llama-3.1-8B" # or local directory with models weights
MODEL_ID = "answerdotai/ModernBERT-large"
STRATEGY = "growth"                        # "growth" | "financial_risk"

common = dict(
    model_id=MODEL_ID,
    strategy=STRATEGY,
    input_dir=INPUT_DATA_DIR,
    output_dir=SENTIMENT_RESULTS_DIR,
    chunk_overlap_tokens=240,
    top_p=0.9,
    docs_per_batch=2,
    save_every=200,
    resume=True,
)

cfg_10q = ExtractionConfig(form_type="10-Q", **common)
cfg_10k = ExtractionConfig(form_type="10-K", **common)

for cfg in (cfg_10q, cfg_10k):
    print(cfg.form_type, "->", cfg.resolve_output_path().name)

10-Q -> growth__ModernBERT-large__10Q.csv
10-K -> growth__ModernBERT-large__10K.csv


## Smoke test

Score a handful of filings first. Watch the log lines for the resolved chunk
size and for any verbalizer words that have no single-token form -- those
contribute nothing to the score.

In [ ]:
from dataclasses import replace

smoke = replace(cfg_10q, limit=5, output_name=f"_smoke_{STRATEGY}_10Q.csv", resume=False)
smoke_result = run_extraction(smoke)
smoke_result

22:45:12 | INFO    | reports | discovered 50 filing(s) under /Users/natalya-vasileva/Desktop/manager_sentiment/input_data
22:45:12 | INFO    | sentiment_extraction | scoring 5 filing(s) with strategy 'growth'


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

W1103 22:45:18.532000 4662 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.58GB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
from reports import ReportDataset
from prompts import normalize_token

backend = ScoringBackend(cfg_10q)
strategy = STRATEGIES[STRATEGY]

item = next(i for i in ReportDataset(sample[sample["FORM_TYPE"] == "10-Q"]) if i.ok)
prompt = strategy.build_prompt(item.text[:20_000], backend.kind, backend.mask_token)

print(item.key)
print("ending:", strategy.ending(backend.kind, backend.mask_token))
print()
for token, prob in backend.top_tokens(prompt, k=20):
    word = normalize_token(token)
    side = (
        "positive" if word in strategy.verbalizer["positive"]
        else "negative" if word in strategy.verbalizer["negative"]
        else ""
    )
    print(f"  {prob:7.4f}  {token!r:20s} {side}")

In [ ]:
del backend
clean_mem()

## Full run

Safe to interrupt and re-run: finished filings are skipped on resume.

In [ ]:
res_10q = run_extraction(cfg_10q)
res_10q.shape

In [ ]:
res_10k = run_extraction(cfg_10k)
res_10k.shape

## Results overview

One row per filing: `positive`, `negative` and `polarity = positive - negative`.

In [ ]:
scores = pd.concat([res_10q, res_10k], ignore_index=True)

cols = ["positive", "negative", "polarity"]
scores.groupby("FORM_TYPE")[cols].describe().T.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for form, group in scores.groupby("FORM_TYPE"):
    axes[0].hist(group["polarity"], bins=60, histtype="step", density=True, label=form)
    axes[1].hist(group["positive"], bins=60, histtype="step", density=True, label=form)
    axes[2].hist(group["negative"], bins=60, histtype="step", density=True, label=form)

for ax, title in zip(axes, ["polarity", "positive", "negative"]):
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Anything that could not be scored, with the reason.
for cfg in (cfg_10q, cfg_10k):
    path = cfg.resolve_output_path()
    failures = path.with_name(path.stem + ".failures.csv")
    if failures.exists():
        frame = pd.read_csv(failures)
        print(f"{cfg.form_type}: {len(frame)} failure(s)")
        print(frame["reason"].str.split(":").str[0].value_counts().to_string())
    else:
        print(f"{cfg.form_type}: no failures")